# Toy hierarchical plug propensity

> **TOY MODEL ON SYNTHETIC COUNTS. NOT A REAL FLEET CALIBRATION.**
> Every number below is invented for teaching. No approved real plug history exists, so nothing
> here supports an accuracy or calibration claim about real cars.

**Question.** On a given eligible day, how likely is each car to plug in, and how many cars in a
small fleet should we expect to plug in tomorrow?

An **eligible day** is one day on which a car was observed and could have plugged in at home. The
outcome is binary: at least one qualifying plug event that day. Counting days throws away time of
day, battery state and multiple sessions per day. The full fleet model handles those; this notebook
only shows the statistical idea of learning a persistent per-car propensity.

**Sections**

1. Synthetic data
2. Simple baseline: a pooled logistic GLM (statsmodels)
3. The hierarchical model and its priors (PyMC)
4. Fitting and diagnostics, with a pass/fail gate
5. Posterior: how the hierarchy shrinks each car
6. Posterior predictive check
7. Predicting tomorrow's fleet count
8. One intact posterior draw into a NumPy forward step
9. Limitations

## Setup

PyMC, ArviZ and statsmodels are imported directly. If one is missing the notebook fails here, on
purpose: a silently skipped fit would look like a finished analysis when nothing ran
(decision 0004 item 17). Install them with `uv sync --group research --group dev`.

PyMC is an offline research tool, not a Streamlit runtime dependency. MCMC belongs in an offline
step; a forecast run should load one coherent posterior draw and use NumPy from then on.

In [1]:
import time

import arviz as az
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pymc as pm
import statsmodels.api as sm
from scipy.special import expit

from axle_studio.ui.style import apply_notebook_style

# One seed for the whole notebook so every number and chart reproduces exactly.
SEED = 20260926
rng = np.random.default_rng(SEED)

# "notebook_connected" embeds only the figure data and loads plotly.js from a CDN, which keeps
# the committed notebook small. GitHub does not run that script, so each chart has a table too.
pio.renderers.default = "notebook_connected"

{"pymc": pm.__version__, "arviz": az.__version__, "statsmodels": sm.__version__}

{'pymc': '6.3.2', 'arviz': '1.3.0', 'statsmodels': '0.15.0'}

## 1. Synthetic data

Eight invented cars. Each row is a sufficient count for this simple likelihood: how many eligible
days the car was observed, and on how many it plugged in. The numbers are fixed, not random, so the
example is easy to inspect. They are chosen to make one point visible: `car-7` has only 6 eligible
days and plugged in on all of them. Its raw rate of 100% is not a believable long-run propensity.

In [2]:
counts = pd.DataFrame(
    {
        "car": [f"car-{i}" for i in range(1, 9)],
        "eligible_days": [18, 20, 16, 25, 12, 30, 6, 22],
        "plugged_days": [13, 8, 4, 17, 9, 21, 6, 10],
    }
)
# Validate the only invariant the likelihood relies on: successes cannot exceed trials.
assert (counts["plugged_days"] >= 0).all()
assert (counts["plugged_days"] <= counts["eligible_days"]).all()

counts["observed_rate"] = counts["plugged_days"] / counts["eligible_days"]
plugged = counts["plugged_days"].to_numpy()
eligible = counts["eligible_days"].to_numpy()
counts.round(3)

,car,eligible_days,plugged_days,observed_rate
0,car-1,18,13,0.722
1,car-2,20,8,0.400
2,car-3,16,4,0.250
3,car-4,25,17,0.680
4,car-5,12,9,0.750
5,car-6,30,21,0.700
6,car-7,6,6,1.000
7,car-8,22,10,0.455


## 2. Simple baseline: pooled logistic GLM

Before any Bayesian machinery, fit the simplest reasonable model: one plug probability shared by
every car. A Binomial GLM with a logit link and only an intercept does exactly that. It is
transparent and fast, and it answers "what is the fleet-wide rate?" well.

Its weakness is the reason for the rest of the notebook. It says every car is the same, so it
cannot tell a habitual plugger from a rare one. The opposite extreme, each car's raw rate on its
own ("no pooling"), over-trusts small samples such as `car-7`.

In [3]:
# Binomial GLM with a two-column response (successes, failures): statsmodels then weights each
# car by its number of eligible days, which is the correct likelihood for these counts.
pooled_glm = sm.GLM(
    np.column_stack([plugged, eligible - plugged]),
    np.ones((len(counts), 1)),
    family=sm.families.Binomial(),
).fit()

pooled_logit = pooled_glm.params[0]
pooled_ci = expit(pooled_glm.conf_int()[0])
pooled_rate = float(expit(pooled_logit))
pd.Series(
    {
        "pooled plug probability": pooled_rate,
        "95% CI low": pooled_ci[0],
        "95% CI high": pooled_ci[1],
        "check: total plugged / total eligible": plugged.sum() / eligible.sum(),
    }
).round(3)

pooled plug probability                  0.591
95% CI low                               0.510
95% CI high                              0.667
check: total plugged / total eligible    0.591
dtype: float64

## 3. The hierarchical model

The hierarchical ("partial pooling") model sits between the two extremes. Each car has its own
persistent propensity, but those propensities are assumed to come from one fleet-wide distribution
that is itself learned from the data. A car with lots of days is mostly judged on its own record; a
car with few days is pulled towards the fleet average. How strongly it is pulled is estimated, not
fixed by hand.

On the log-odds (logit) scale:

$$
\text{logit}(p_i) = \mu + \sigma z_i, \qquad z_i \sim \text{Normal}(0, 1), \qquad
y_i \sim \text{Binomial}(n_i, p_i)
$$

**Priors, in plain terms**

- `fleet_logit` $\mu \sim \text{Normal}(0, 1.5)$: the typical car's log-odds of plugging. Centred on
  a 50% rate; about 95% of the prior mass puts the typical rate between roughly 5% and 95%. It rules
  out only near-certain extremes.
- `car_sd` $\sigma \sim \text{HalfNormal}(1)$: how different cars are from each other on the logit
  scale. Values near 0 mean near-identical cars (pooled model); values near 2 mean cars range from
  rare to near-constant pluggers. It gently favours moderate differences.
- `z` $\sim \text{Normal}(0, 1)$: each car's standardised offset from the typical car.

**Why this parameterisation.** Writing $\text{logit}(p_i) = \mu + \sigma z_i$ ("non-centred") rather
than drawing $\text{logit}(p_i) \sim \text{Normal}(\mu, \sigma)$ directly avoids a funnel-shaped
posterior when $\sigma$ is small, which is the usual cause of divergences in hierarchical models with
few groups. It is the same model; only the geometry the sampler sees changes.

The logit-normal form replaces an earlier Beta/Gamma hierarchy. Both are standard; the logit form
has a direct link to the logistic GLM baseline (the pooled model is the special case $\sigma = 0$)
and extends naturally to covariates later.

In [4]:
with pm.Model(coords={"car": counts["car"].to_numpy()}) as plug_model:
    fleet_logit = pm.Normal("fleet_logit", mu=0.0, sigma=1.5)
    car_sd = pm.HalfNormal("car_sd", sigma=1.0)
    z = pm.Normal("z", mu=0.0, sigma=1.0, dims="car")
    # Non-centred: sample standard offsets z, then scale. Avoids the small-sigma funnel.
    p_car = pm.Deterministic("p_car", pm.math.invlogit(fleet_logit + car_sd * z), dims="car")
    pm.Binomial("plugged_days", n=eligible, p=p_car, observed=plugged, dims="car")

# Prior predictive check: what fleet-typical rates do the priors alone imply, before seeing data?
with plug_model:
    prior = pm.sample_prior_predictive(draws=2000, random_seed=SEED)
prior_typical_rate = expit(prior.prior["fleet_logit"].values.ravel())
pd.Series(
    np.quantile(prior_typical_rate, [0.05, 0.25, 0.5, 0.75, 0.95]),
    index=["5%", "25%", "50%", "75%", "95%"],
    name="prior typical plug rate",
).round(3)

Sampling: [car_sd, fleet_logit, plugged_days, z]


5%     0.078
25%    0.273
50%    0.511
75%    0.742
95%    0.918
Name: prior typical plug rate, dtype: float64

The prior alone allows almost any typical rate, so the data, not the prior, will drive the answer.

## 4. Fit and diagnostics

NUTS (the default PyMC sampler) runs four independent chains. Four chains let us compare them: if
they disagree, the sampler has not converged. The settings are modest because this model is tiny; the
fit takes a few seconds.

**How to read the diagnostics**

- **R-hat** compares the spread within each chain to the spread between chains. At convergence it is
  1.00. We require every value below 1.01 (the current recommended threshold).
- **ESS (effective sample size)** is how many independent draws the correlated MCMC draws are worth.
  `ess_bulk` covers the centre of the distribution, `ess_tail` the 5% and 95% tails. We require at
  least 400 (100 per chain) for every parameter.
- **Divergences** flag places where the sampler's numerical integration broke down, which usually
  means part of the posterior was missed. We require zero. If divergences appear, the fix is to
  reparameterise (as done above), not to loosen this gate.

In [5]:
start = time.perf_counter()
with plug_model:
    idata = pm.sample(
        draws=2000,
        tune=1500,
        chains=4,
        cores=4,
        target_accept=0.9,  # smaller steps than the 0.8 default; cheap at this model size
        random_seed=SEED,
        progressbar=False,
    )
fit_seconds = time.perf_counter() - start

diagnostics = az.summary(
    idata, var_names=["fleet_logit", "car_sd", "p_car"], kind="diagnostics", round_to="none"
)[["r_hat", "ess_bulk", "ess_tail"]]
divergences = int(idata.sample_stats["diverging"].sum())
print(f"Fit time: {fit_seconds:.1f} s; divergences: {divergences}")
diagnostics.round(4)

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [fleet_logit, car_sd, z]


Sampling 4 chains for 1_500 tune and 2_000 draw iterations (6_000 + 8_000 draws total) took 1 seconds.


Fit time: 1.8 s; divergences: 0


,r_hat,ess_bulk,ess_tail
fleet_logit,1.0007,2661.4610,3024.6206
car_sd,1.0008,1899.7724,2850.1201
p_car[car-1],1.0014,9323.8606,6758.9815
p_car[car-2],1.0003,7495.6770,5308.7049
p_car[car-3],1.0004,4977.0427,3768.2567
p_car[car-4],0.9998,10424.9513,6961.8213
p_car[car-5],1.0007,7494.6810,6129.0357
p_car[car-6],0.9998,9120.6041,6608.2230
p_car[car-7],1.0007,3896.7409,4477.5527
p_car[car-8],1.0000,9688.1724,6920.0723


In [6]:
# Pass/fail gate. Everything after this cell interprets the posterior, so it must not run on a
# fit that has not converged. A failure raises and stops the notebook.
MAX_R_HAT = 1.01
MIN_ESS = 400

assert divergences == 0, f"{divergences} divergent transitions; reparameterise, do not loosen"
assert np.isfinite(diagnostics.to_numpy(dtype=float)).all(), "non-finite diagnostic"
assert (diagnostics["r_hat"] < MAX_R_HAT).all(), diagnostics["r_hat"].max()
assert (diagnostics[["ess_bulk", "ess_tail"]] >= MIN_ESS).all().all(), diagnostics.min()

print(
    f"PASS: max R-hat {diagnostics['r_hat'].max():.4f} < {MAX_R_HAT}, "
    f"min ESS {diagnostics[['ess_bulk', 'ess_tail']].min().min():.0f} >= {MIN_ESS}, "
    f"0 divergences."
)

PASS: max R-hat 1.0014 < 1.01, min ESS 1900 >= 400, 0 divergences.


## 5. Posterior: shrinkage towards the fleet

The table compares three answers for each car: its raw observed rate (no pooling), the single
pooled GLM rate, and the hierarchical posterior mean with a 90% credible interval.

Read it as follows. Cars with many days (`car-6`, `car-4`) stay close to their own raw rate. `car-7`,
with 6 out of 6, is pulled well below 100% because six days are weak evidence. `car-3`, the rarest
plugger, is pulled up towards the fleet. This is the practical benefit of the hierarchy: sensible
estimates for cars with little history, without pretending all cars are the same.

In [7]:
post_p = idata.posterior["p_car"]
shrinkage = pd.DataFrame(
    {
        "eligible_days": eligible,
        "observed_rate": counts["observed_rate"].to_numpy(),
        "pooled_glm": pooled_rate,
        "hierarchical_mean": post_p.mean(("chain", "draw")).values,
        "hier_5%": post_p.quantile(0.05, ("chain", "draw")).values,
        "hier_95%": post_p.quantile(0.95, ("chain", "draw")).values,
    },
    index=counts["car"],
)

fleet_typical = expit(idata.posterior["fleet_logit"].values.ravel())
print(
    f"Typical-car plug rate: {fleet_typical.mean():.3f} "
    f"(90% interval {np.quantile(fleet_typical, 0.05):.3f} to "
    f"{np.quantile(fleet_typical, 0.95):.3f}); "
    f"car_sd posterior mean {float(idata.posterior['car_sd'].mean()):.2f} on the logit scale"
)
shrinkage.round(3)

Typical-car plug rate: 0.607 (90% interval 0.470 to 0.745); car_sd posterior mean 0.86 on the logit scale


,eligible_days,observed_rate,pooled_glm,hierarchical_mean,hier_5%,hier_95%
car,,,,,,
car-1,18,0.722,0.591,0.684,0.526,0.831
car-2,20,0.400,0.591,0.457,0.287,0.618
car-3,16,0.250,0.591,0.365,0.184,0.556
car-4,25,0.680,0.591,0.660,0.517,0.796
car-5,12,0.750,0.591,0.688,0.507,0.856
car-6,30,0.700,0.591,0.677,0.545,0.801
car-7,6,1.000,0.591,0.769,0.550,0.953
car-8,22,0.455,0.591,0.495,0.333,0.649


In [8]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=shrinkage.index,
        y=shrinkage["hierarchical_mean"],
        error_y=dict(
            type="data",
            symmetric=False,
            array=shrinkage["hier_95%"] - shrinkage["hierarchical_mean"],
            arrayminus=shrinkage["hierarchical_mean"] - shrinkage["hier_5%"],
        ),
        mode="markers",
        name="Hierarchical mean (90% interval)",
    )
)
fig.add_trace(
    go.Scatter(
        x=shrinkage.index,
        y=shrinkage["observed_rate"],
        mode="markers",
        marker=dict(symbol="x", size=10),
        name="Observed rate (no pooling)",
    )
)
fig.add_hline(y=pooled_rate, line_dash="dash", annotation_text="Pooled GLM rate")
fig.update_layout(
    title="Toy synthetic data: hierarchical estimates shrink towards the fleet",
    yaxis_title="P(plug on an eligible day)",
    yaxis_range=[0, 1.05],
)
apply_notebook_style(fig, height=400)
fig.show()

## 6. Posterior predictive check

A posterior predictive check asks: if the fitted model were true, would it generate data like the
data we saw? For every posterior draw we simulate a fresh count of plugged days for each car, using
that car's real number of eligible days. If the observed counts sit comfortably inside the simulated
ranges, the model is at least not contradicted by its own training data.

This is a sanity check, not validation. A model can pass it and still predict new data poorly, and
with synthetic data there is no real out-of-sample test to run.

In [9]:
with plug_model:
    ppc = pm.sample_posterior_predictive(idata, random_seed=SEED, progressbar=False)

replicated = ppc.posterior_predictive["plugged_days"]
ppc_table = pd.DataFrame(
    {
        "observed": plugged,
        "replicated_5%": replicated.quantile(0.05, ("chain", "draw")).values,
        "replicated_median": replicated.quantile(0.5, ("chain", "draw")).values,
        "replicated_95%": replicated.quantile(0.95, ("chain", "draw")).values,
    },
    index=counts["car"],
)
ppc_table["inside_90%"] = ppc_table["observed"].between(
    ppc_table["replicated_5%"], ppc_table["replicated_95%"]
)
print(f"{ppc_table['inside_90%'].sum()} of {len(ppc_table)} observed counts inside the 90% range")
ppc_table

Sampling: [plugged_days]


8 of 8 observed counts inside the 90% range


,observed,replicated_5%,replicated_median,replicated_95%,inside_90%
car,,,,,
car-1,13,8.0,13.0,16.0,True
car-2,8,4.0,9.0,14.0,True
car-3,4,2.0,6.0,10.0,True
car-4,17,11.0,17.0,21.0,True
car-5,9,5.0,8.0,11.0,True
car-6,21,15.0,20.0,26.0,True
car-7,6,2.0,5.0,6.0,True
car-8,10,6.0,11.0,16.0,True


In [10]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ppc_table.index,
        y=ppc_table["replicated_median"],
        error_y=dict(
            type="data",
            symmetric=False,
            array=ppc_table["replicated_95%"] - ppc_table["replicated_median"],
            arrayminus=ppc_table["replicated_median"] - ppc_table["replicated_5%"],
        ),
        mode="markers",
        name="Replicated plugged days (median, 90% range)",
    )
)
fig.add_trace(
    go.Scatter(
        x=ppc_table.index,
        y=ppc_table["observed"],
        mode="markers",
        marker=dict(symbol="x", size=10),
        name="Observed plugged days",
    )
)
fig.update_layout(
    title="Toy synthetic data: posterior predictive check",
    yaxis_title="Plugged days",
)
apply_notebook_style(fig, height=400)
fig.show()

## 7. Predicting tomorrow's fleet count

For each posterior draw we flip one coin per car with that draw's `p_car`, then add the cars up
**within the draw**. Summing within each draw keeps the cars' shared uncertainty about the fleet
together; adding separately computed per-car summaries (or percentiles) would get the spread wrong.

In [11]:
# Flatten chains x draws into one axis of joint posterior samples, shape (samples, cars).
p_samples = post_p.stack(sample=("chain", "draw")).transpose("sample", "car").values
tomorrow = rng.binomial(1, p_samples)  # one Bernoulli per car per posterior draw
fleet_count = tomorrow.sum(axis=1)  # aggregate within each draw first, then summarise

fleet_distribution = (
    pd.Series(fleet_count)
    .value_counts(normalize=True)
    .reindex(range(len(counts) + 1), fill_value=0.0)
    .rename_axis("cars plugged tomorrow")
    .rename("probability")
)
assert np.isclose(fleet_distribution.sum(), 1.0)
print(
    f"Expected cars plugged tomorrow: {fleet_count.mean():.2f}; "
    f"90% interval {np.quantile(fleet_count, 0.05):.0f} to {np.quantile(fleet_count, 0.95):.0f}"
)
fleet_distribution.round(3).to_frame().T

Expected cars plugged tomorrow: 4.79; 90% interval 3 to 7


cars plugged tomorrow,0,1,2,3,4,5,6,7,8
probability,0.0,0.008,0.038,0.123,0.236,0.296,0.2,0.084,0.015


In [12]:
fig = go.Figure(go.Bar(x=fleet_distribution.index, y=fleet_distribution.values))
fig.update_layout(
    title="Toy synthetic data: predicted number of the 8 cars plugging in tomorrow",
    xaxis_title="Cars plugged tomorrow",
    yaxis_title="Probability",
)
apply_notebook_style(fig, height=350)
fig.show()

## 8. One intact posterior draw into a NumPy forward step

This mirrors the production boundary. An explicit forecast run selects **one** whole posterior draw
at the start (all cars' `p_car` values from the same draw, so their joint structure is kept), then
simulates forward with NumPy. `p_car` is a persistent trait: it is fixed for the run, not resampled
each day.

The small function below only samples a daily plug event and carries a toy battery stock forward.
It does not sample arrival state of charge. In the full model, arrival SoC is the output of
away/home/plugged/charging transitions and battery energy in and out.

In [13]:
def forward_toy_days(p_car: np.ndarray, *, seed: int, days: int = 3) -> pd.DataFrame:
    """Simulate daily plug events and a toy battery stock for each car.

    p_car: persistent per-car plug probability from one posterior draw (0 to 1).
    Returns one row per car-day with opening and closing battery energy in kWh.
    The 30 kWh start, +4 kWh per plugged day and 60 kWh cap are illustrative only.
    """
    local_rng = np.random.default_rng(seed)
    battery_kwh = np.full(len(p_car), 30.0)
    rows = []
    for day in range(1, days + 1):
        plugged_today = local_rng.binomial(1, p_car)
        opening = battery_kwh.copy()
        # Stock flow: the closing stock is the opening stock plus energy added, never re-sampled.
        battery_kwh = np.minimum(60.0, battery_kwh + plugged_today * 4.0)
        for i, car in enumerate(counts["car"]):
            rows.append(
                {
                    "day": day,
                    "car": car,
                    "plugged": int(plugged_today[i]),
                    "opening_battery_kwh": opening[i],
                    "closing_battery_kwh": battery_kwh[i],
                }
            )
    return pd.DataFrame(rows)


draw_index = int(rng.integers(len(p_samples)))  # a real run records this index with its results
forward = forward_toy_days(p_samples[draw_index], seed=SEED)
print(f"Using posterior draw {draw_index}")
forward.head(8)

Using posterior draw 1398


,day,car,plugged,opening_battery_kwh,closing_battery_kwh
0,1,car-1,0,30.0,30.0
1,1,car-2,1,30.0,34.0
2,1,car-3,0,30.0,30.0
3,1,car-4,0,30.0,30.0
4,1,car-5,0,30.0,30.0
5,1,car-6,1,30.0,34.0
6,1,car-7,1,30.0,34.0
7,1,car-8,1,30.0,34.0


## 9. Limitations

- **Synthetic and toy.** The counts are invented. Nothing here is calibrated to, or validated
  against, real cars.
- **Eight cars.** With few groups the fleet-level parameters, especially `car_sd`, are uncertain and
  somewhat sensitive to their priors. A real fit would report a prior sensitivity check.
- **Exchangeable days.** The Binomial likelihood treats every eligible day of a car as an
  independent coin flip with the same probability. Real behaviour has weekday patterns, habits
  that drift, and dependence on battery level and tariff; none of that is modelled.
- **No covariates.** Car type, commute length or weekday could enter the logit linearly; they are left
  out to keep the idea clear.
- **In-sample checks only.** The posterior predictive check tests internal consistency. A real use
  needs held-out data and calibration checks.

Before any real plug history is used, the lead must approve the observation definition, data
provenance, privacy and licence, the cohort mapping and the offline fit/export path. This notebook
must not run from Streamlit and must not be presented as fleet calibration.